# 01 - Preprocessing

Cleans the three NCC-2 sensor captures and writes processed CSVs.
The ordinal encoder and scaler are fit on Sensor 1 and reused for Sensors 2 and 3,
so all sensors live in the same feature space.

Expected layout: `data/raw/sensor{1,2,3}/sensor{1,2,3}.binetflow`

In [1]:
import sys
sys.path.append("..")

import os
import joblib
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from src.common import RAW_DIR, PROCESSED_DIR, TARGET, CATEGORICAL_COLS

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [2]:
DROP_COLS = ["Label", "BotnetName", "SensorId", "SrcAddr", "DstAddr"]


def clean(df):
    """Fill structural NaNs, derive time features, drop identifiers/leaky labels."""
    df = df.copy()
    for col in ["Sport", "Dport", "State"]:
        df[col] = df[col].fillna("Unknown")
    for col in ["sTos", "dTos"]:
        df[col] = df[col].fillna(0)

    ts = pd.to_datetime(df["StartTime"])
    df["Hour"], df["Minute"], df["Second"] = ts.dt.hour, ts.dt.minute, ts.dt.second
    df = df.drop(columns=["StartTime"] + DROP_COLS)

    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    return X, y

In [3]:
encoder, scaler, numerical_cols = None, None, None

for i in [1, 2, 3]:
    raw = pd.read_csv(f"{RAW_DIR}/sensor{i}/sensor{i}.binetflow")
    X, y = clean(raw)

    if encoder is None:  # fit on Sensor 1 only
        numerical_cols = [c for c in X.columns if c not in CATEGORICAL_COLS]
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        scaler = StandardScaler()
        encoder.fit(X[CATEGORICAL_COLS].astype(str))
        scaler.fit(X[numerical_cols])
        joblib.dump(encoder, f"{PROCESSED_DIR}/encoder.pkl")
        joblib.dump(scaler, f"{PROCESSED_DIR}/scaler.pkl")

    X[CATEGORICAL_COLS] = encoder.transform(X[CATEGORICAL_COLS].astype(str))
    X[numerical_cols] = scaler.transform(X[numerical_cols])

    out = X.copy()
    out[TARGET] = y
    out = out.dropna()
    out.to_csv(f"{PROCESSED_DIR}/sensor{i}_processed.csv", index=False, float_format="%.6f")
    print(f"sensor{i}: {out.shape}")

sensor1: (4895158, 15)
sensor2: (5998133, 15)
sensor3: (3885792, 15)
